# Tut 2 
## THD Estimation of a Measured Blocked Rotor Current Using a Digital FIR Filter Bank

This report investigates the total harmonic distortion (THD) of a measured blocked rotor current signal using a digital FIR filter bank. The measured signal is first examined in both the time domain and the frequency domain. After this, three FIR filter design methods are applied to estimate the harmonic content of the signal. The resulting THD values are then compared with an FFT based reference value.

The report is presented as a single study in which all three FIR design methods are evaluated using the same harmonic estimation procedure. This allows a fair comparison between the methods and makes it easier to assess how the choice of filter design affects the final THD result.

The main objectives of the tutorial are:

1. to design and implement an FIR filter from first principles,
2. to apply the frequency sampling method,
3. to use the Parks McClellan method through scipy.signal.remez,
4. to estimate THD from the measured data,
5. to verify the result using the FFT.

The report begins with the definition of THD, followed by an inspection of the measured signal and the design of the FIR filters. It concludes with a comparison between the FIR based THD estimates and the FFT based verification result.

## 1. Introduction and Objective

The aim of this assignment is to estimate the total harmonic distortion, or THD, of the measured blocked rotor current stored in BlockedRotorTest.csv. This is done by using a filter bank made up of FIR band pass filters. Each filter is used to isolate a specific part of the signal, such as the fundamental component or one of the harmonics. The rms values of these components are then used to calculate the THD.

To meet the tutorial requirements, three FIR design methods are used in the analysis:

1. the window method,
2. the frequency sampling method,
3. the Parks McClellan equiripple method using `remez`.

Once the FIR based THD estimates have been found, the result is checked using an FFT based harmonic analysis. This provides a second result that can be used to verify the filter bank estimate.

## 2. THD Background

In power quality studies, the most common definition is total harmonic distortion relative to the fundamental component, denoted by $THD_F$:

$$
THD_F = \frac{\sqrt{\sum_{n=2}^{\infty} I_n^2}}{I_1}
$$

where $I_1$ is the rms value of the fundamental component and $I_n$ is the rms value of the $n$ th harmonic component.

A digital filter bank can be used to estimate these rms values by isolating each harmonic with a narrow band pass filter. Once the filtered outputs have been obtained, the rms value of each output can be calculated directly in the time domain. The FFT is then used as an independent reference method to verify the final THD estimate.

## 3. Signal Import and Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the measured CSV data
# The file contains header information, so the first six rows are skipped
df = pd.read_csv("BlockedRotorTest.csv", skiprows=6, header=None)

# Keep the columns required for the analysis
df = df.iloc[:, :3].copy()
df.columns = ["time", "current_V", "voltage_V"]

# Convert the imported values to numeric form and discard invalid rows
df["time"] = pd.to_numeric(df["time"], errors="coerce")
df["current_V"] = pd.to_numeric(df["current_V"], errors="coerce")
df["voltage_V"] = pd.to_numeric(df["voltage_V"], errors="coerce")
df = df.dropna().reset_index(drop=True)

# Store the time and current columns as NumPy arrays
t = df["time"].to_numpy()
x = df["current_V"].to_numpy()

# Determine the average sampling interval and sampling frequency
dt = np.mean(np.diff(t))
fs = 1 / dt

print(f"Number of samples: {len(x)}")
print(f"Sampling interval dt = {dt:.9f} s")
print(f"Sampling frequency fs = {fs:.3f} Hz")

# Plot the first part of the measured signal for an initial time domain view
plt.figure(figsize=(10, 4))
plt.plot(t[:3000], x[:3000])
plt.xlabel("Time (s)")
plt.ylabel("Current sensor output (V)")
plt.title("Measured current signal (first segment)")
plt.grid(True)
plt.show()

The measured data are first imported from the CSV file, after which a time axis is obtained from the recorded sampling interval. The mean value of the signal is then removed so that any DC offset does not influence the later spectral analysis. The zero mean signal is used for the remainder of the report.

## 4. Initial Spectral Inspection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Remove the DC component so that the spectrum reflects only the AC content
x_ac = x - np.mean(x)

N = len(x_ac)
X = np.fft.rfft(x_ac)
f = np.fft.rfftfreq(N, d=dt)

# Convert the FFT magnitude to a one sided amplitude spectrum
mag = 2 * np.abs(X) / N
mag[0] = np.abs(X[0]) / N # DC term correction

# Plot the low frequency part of the spectrum where the main harmonics appear
plt.figure(figsize=(12, 5))
plt.plot(f, mag)
plt.xlim(0, 1000)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("FFT magnitude spectrum of measured current signal")
plt.grid(True)
plt.show()

# Identify the strongest non zero frequency component as the estimated fundamental
search_limit = np.where(f <= 500)[0][-1]
peak_index = np.argmax(mag[1:search_limit]) + 1
f1_est = f[peak_index]

print(f"Estimated fundamental frequency = {f1_est:.4f} Hz")

The FFT of the measured signal is inspected at integer multiples of the fundamental frequency so that the dominant harmonics can be identified. This provides a practical basis for choosing the harmonic range that will later be used in the filter bank.

In [ ]:
# Use the detected fundamental frequency in the following calculations
f1 = f1_est

# Select the number of harmonics to inspect during the preliminary analysis
max_harmonic = 15

harmonic_freqs = []
harmonic_mags = []

for n in range(1, max_harmonic + 1):
 target_freq = n * f1
 idx = np.argmin(np.abs(f - target_freq))
 harmonic_freqs.append(f[idx])
 harmonic_mags.append(mag[idx])

print("Harmonic inspection:")
for n, (freq_n, mag_n) in enumerate(zip(harmonic_freqs, harmonic_mags), start=1):
 print(f"Harmonic {n:2d}: frequency = {freq_n:8.3f} Hz, magnitude = {mag_n:.6f}")

# Display the harmonic magnitudes obtained from the preliminary FFT
plt.figure(figsize=(12, 5))
plt.bar(range(1, max_harmonic + 1), harmonic_mags)
plt.xlabel("Harmonic number")
plt.ylabel("Magnitude")
plt.title("Estimated harmonic magnitudes from FFT")
plt.grid(True, axis='y')
plt.show()

The FFT spectrum shows that the fundamental component is located close to 50 Hz. Further components are visible at integer multiples of this frequency, which confirms that the signal contains harmonic distortion. The most significant lower order harmonics are used to guide the later filter bank design.

At this stage, the FFT is used only for initial inspection of the signal. Its purpose here is to identify the approximate fundamental frequency and the main low order harmonics present in the measured current. The formal FFT based THD verification is carried out later, after all three FIR design methods have been presented.

## 5. Selection of Harmonic Range

In [ ]:
highest_possible_harmonic = int((fs / 2) // f1)
print(f"Highest harmonic below Nyquist = {highest_possible_harmonic}")

Although the Nyquist limit allows a much higher harmonic order to be analysed, the preliminary FFT shows that the dominant distortion components are concentrated in the lower order harmonics. For that reason, the filter bank is designed around the fundamental and the first ten harmonics during the initial discussion.

The FFT analysis identified the fundamental component at approximately $49.9023\ \text{Hz}$, which is consistent with a nominal 50 Hz laboratory supply. Even though the sampling frequency would allow much higher harmonic orders below Nyquist, the strongest spectral content is concentrated in the lower harmonics. The final filter bank is therefore limited to the fundamental and the first 15 harmonics.

$$
f_n = n f_1
$$

$$
f_c = f_1 \approx 49.9023\ \text{Hz}
$$

## 6. Common FIR Filter Bank Strategy

The same filter bank method is used throughout this report. The input signal is split into the fundamental component and a set of selected harmonics. For each harmonic, a narrow band pass FIR filter is designed around the centre frequency, with $f_c = n f_1$. The rms value of the filtered signal is then taken as the estimated rms value of that harmonic.

The key idea is that the estimation procedure stays the same for all three methods. The only difference is how the FIR filter coefficients are obtained.

This means the methods are compared on a fair basis. Any difference in the final THD estimate can therefore be linked mainly to the filter design method itself, and not to changes in the overall estimation process.

## 7. FIR Design Method 1: Window Method

The window method starts from the ideal impulse response and then truncates it by means of a chosen window. For a band pass filter, the ideal response can be formed from the difference between two ideal low pass responses. This makes the method suitable for the harmonic filter bank required for THD estimation.

The following cells first demonstrate the design for a representative filter centred on the fundamental component. The same design principle is then extended to the full harmonic bank.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fundamental frequency settings
f1 = 49.9023 # estimated from the preliminary FFT inspection
bw = 20 # band-pass width in Hz for the representative design
M = 401 # odd length to obtain a symmetric linear phase FIR filter

# Helper function for the ideal low pass impulse response
def ideal_lowpass(wc, m):
 h = np.zeros_like(m, dtype=float)
 for i, k in enumerate(m):
 if k == 0:
 h[i] = wc / np.pi
 else:
 h[i] = np.sin(wc * k) / (np.pi * k)
 return h

def design_bandpass_fir(fc, bw, fs, M):
 """
 Band pass FIR design using the window method.
 The ideal band-pass response is obtained by subtracting two ideal
 low pass responses and then truncating with a Hamming window.
 """
 n = np.arange(M)
 alpha = (M - 1) / 2
 m = n - alpha

 f_low = max(0.0, fc - bw / 2)
 f_high = min(fs / 2, fc + bw / 2)

 w1 = 2 * np.pi * f_low / fs
 w2 = 2 * np.pi * f_high / fs

 h_lp2 = ideal_lowpass(w2, m)
 h_lp1 = ideal_lowpass(w1, m)
 h_bp_ideal = h_lp2 - h_lp1

 window = 0.54 - 0.46 * np.cos(2 * np.pi * n / (M - 1))
 return h_bp_ideal * window

# Design a representative band pass filter centred on the fundamental
h_bp = design_bandpass_fir(f1, bw, fs, M)
f_low = f1 - bw/2
f_high = f1 + bw/2

print(f"Band pass filter designed for {f_low:.2f} Hz to {f_high:.2f} Hz")
print(f"Filter length = {M}")


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(h_bp)
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.title("Impulse response of FIR bandpass filter for the fundamental")
plt.grid(True)
plt.show()

The frequency response of the representative window designed FIR filter is evaluated to confirm that the passband is centred on the fundamental component and that frequencies outside the selected range are attenuated. Once this behaviour has been confirmed, the same design approach is used later in the complete harmonic filter bank.

In [ ]:
# Obtain the filter frequency response from the FFT of the coefficients
N_resp = 16384
H = np.fft.rfft(h_bp, n=N_resp)
f_resp = np.fft.rfftfreq(N_resp, d=1/fs)

H_mag = np.abs(H)
H_mag_db = 20 * np.log10(H_mag / np.max(H_mag) + 1e-12)

plt.figure(figsize=(12, 5))
plt.plot(f_resp, H_mag_db)
plt.xlim(0, 300)
plt.ylim(-100, 5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.title("Frequency response of the FIR bandpass filter")
plt.grid(True)
plt.show()

The designed FIR filter is then applied to the measured current signal by discrete time convolution. Because the filter has linear phase and symmetric coefficients, the resulting waveform provides an estimate of the signal content around the selected passband. The rms value of this output is used as the estimate of the corresponding harmonic component.

In [ ]:
# Apply the FIR filter to the signal by discrete time convolution
y1 = np.convolve(x_ac, h_bp, mode='same')

# Compare the original signal with the filtered fundamental component
plt.figure(figsize=(12, 5))
plt.plot(t[:3000], x_ac[:3000], label="Original current signal", alpha=0.6)
plt.plot(t[:3000], y1[:3000], label="Filtered fundamental component", linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (V)")
plt.title("Original signal and filtered fundamental component")
plt.grid(True)
plt.legend()
plt.show()

The rms value of the filtered output is used as the estimate of the harmonic component:

$$
I_n = \sqrt{\frac{1}{N}\sum_{k=0}^{N-1} y_n^2[k]}
$$

For the first filter, this gives the estimate of the fundamental component $I_1$.

In [ ]:
I1_est = np.sqrt(np.mean(y1**2))
print(f"Estimated fundamental RMS from filter = {I1_est:.6f}")

## 8. FIR Design Method 2: Frequency Sampling Method

In the frequency sampling method, the desired frequency response is specified directly on a uniform frequency grid. The time domain FIR coefficients are then obtained from the inverse discrete Fourier transform. The same band pass specification used in the window method is retained so that the three approaches can be compared on a consistent basis.

### 8.1 Design Principle

For a filter length $M$, the desired response is sampled at the discrete frequencies

$$
\omega_k = \frac{2\pi k}{M}, \qquad k=0,1,\ldots,M-1
$$

and the coefficients are obtained from the inverse DFT,

$$
h[n] = \frac{1}{M}\sum_{k=0}^{M-1} H[k] e^{j2\pi kn/M}.
$$

To obtain a linear phase band pass FIR filter, the desired response is given unit magnitude in the selected passband and multiplied by a linear phase term $e^{-j\omega_k \alpha}$, where $\alpha = (M-1)/2$.

In [ ]:

def design_bandpass_freq_sampling(fc, bw, fs, M):
 """Band pass FIR design using the frequency sampling method."""
 alpha = (M - 1) / 2
 k = np.arange(M)
 f_k = k * fs / M

 f_low = fc - bw / 2
 f_high = fc + bw / 2

 # Define the desired magnitude response over the discrete frequency grid
 pass_pos = (f_k >= f_low) & (f_k <= f_high)
 pass_neg = (f_k >= fs - f_high) & (f_k <= fs - f_low)
 magnitude = np.where(pass_pos | pass_neg, 1.0, 0.0)

 # Apply the linear phase term so that the filter is centred correctly in time
 Hk = magnitude * np.exp(-1j * 2 * np.pi * k * alpha / M)

 # Convert the sampled frequency response back to the time domain
 h = np.fft.ifft(Hk)
 h = np.real_if_close(h, tol=1e5).real
 return h

M_fs = 401
bw_fs = 20
fc_demo = f1

h_fs = design_bandpass_freq_sampling(fc_demo, bw_fs, fs, M_fs)
print(f"Frequency-sampling filter generated with {len(h_fs)} taps.")

### 8.2 Coefficient Calculation and Impulse Response

The coefficients below are generated from the sampled desired response for a representative band pass filter centred on the fundamental component.

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(h_fs)
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.title("Impulse response of the frequency-sampling band-pass filter")
plt.grid(True)
plt.show()

### 8.3 Frequency Response

The frequency response is inspected to verify that the passband is centred on the fundamental frequency and that nearby out of band components are attenuated.

In [ ]:

N_resp = 16384
H_fs = np.fft.rfft(h_fs, n=N_resp)
f_resp = np.fft.rfftfreq(N_resp, d=1/fs)
H_fs_db = 20 * np.log10(np.abs(H_fs) / np.max(np.abs(H_fs)) + 1e-12)

plt.figure(figsize=(12, 5))
plt.plot(f_resp, H_fs_db)
plt.xlim(0, 300)
plt.ylim(-100, 5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.title("Frequency response of the frequency-sampling FIR filter")
plt.grid(True)
plt.show()

### 8.4 Harmonic Estimation Using This Filter Bank

The same convolution based harmonic estimation procedure used for the window method is now applied with the frequency sampling coefficients for the representative fundamental filter.

In [ ]:

y1_fs = np.convolve(x_ac, h_fs, mode='same')
I1_fs_demo = np.sqrt(np.mean(y1_fs**2))

plt.figure(figsize=(12, 5))
plt.plot(t[:3000], x_ac[:3000], label="Original current signal", alpha=0.6)
plt.plot(t[:3000], y1_fs[:3000], label="Filtered fundamental component", linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (V)")
plt.title("Original signal and frequency-sampling filtered fundamental")
plt.grid(True)
plt.legend()
plt.show()

print(f"Estimated fundamental RMS from frequency-sampling filter = {I1_fs_demo:.6f}")

### 8.5 Comment

The frequency sampling method offers a direct way to obtain a linear phase FIR filter from a selected discrete frequency specification. In this report, it produced a practical band pass response and a corresponding THD estimate using the same harmonic estimation approach as the other methods. Its main limitation is that the final response depends strongly on the chosen frequency grid and filter length. As a result, narrow harmonic components can only be separated effectively when the filter is long enough.

## 9. FIR Design Method 3: Equiripple Method

The Parks McClellan method produces an equiripple minimax FIR filter. This is the only part of the report where a built in FIR design routine is used, which is consistent with the tutorial instruction that `scipy.signal.remez` may be used as the allowed exception.

### 9.1 Design Principle

The equiripple design solves a minimax approximation problem. Instead of matching the ideal response exactly at every frequency, the algorithm distributes the approximation error so that the maximum weighted error is minimised over the passband and stopband regions.

For the present problem, the desired response is a narrow band pass response around the harmonic of interest.

In [ ]:

from scipy import signal

def design_bandpass_remez(fc, bw, fs, M, transition=4.0):
 """Band pass FIR design using Parks McClellan / remez."""
 f_low = fc - bw / 2
 f_high = fc + bw / 2

 left_stop = max(0.0, f_low - transition)
 right_stop = min(fs / 2, f_high + transition)

 bands = [0.0, left_stop, f_low, f_high, right_stop, fs / 2]
 desired = [0, 1, 0]
 weight = [1, 1, 1]
 return signal.remez(numtaps=M, bands=bands, desired=desired, weight=weight, fs=fs)

M_remez = 401
bw_remez = 20
h_remez = design_bandpass_remez(fc_demo, bw_remez, fs, M_remez, transition=8.0)
print(f"Equiripple (remez) filter generated with {len(h_remez)} taps.")

### 9.2 Coefficient Calculation and Impulse Response

The impulse response below corresponds to the equiripple band pass filter designed for the fundamental component.

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(h_remez)
plt.xlabel("Sample index")
plt.ylabel("Amplitude")
plt.title("Impulse response of the equiripple band-pass filter")
plt.grid(True)
plt.show()

### 9.3 Frequency Response

The equiripple response is inspected in the same way as the previous designs so that all three methods can be compared under the same conditions.

In [ ]:

H_remez = np.fft.rfft(h_remez, n=N_resp)
H_remez_db = 20 * np.log10(np.abs(H_remez) / np.max(np.abs(H_remez)) + 1e-12)

plt.figure(figsize=(12, 5))
plt.plot(f_resp, H_remez_db)
plt.xlim(0, 300)
plt.ylim(-100, 5)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.title("Frequency response of the equiripple FIR filter")
plt.grid(True)
plt.show()

### 9.4 Harmonic Estimation Using This Filter Bank

The designed equiripple filter is applied to the signal to estimate the fundamental component in the same manner as before.

In [ ]:

y1_remez = np.convolve(x_ac, h_remez, mode='same')
I1_remez_demo = np.sqrt(np.mean(y1_remez**2))

plt.figure(figsize=(12, 5))
plt.plot(t[:3000], x_ac[:3000], label="Original current signal", alpha=0.6)
plt.plot(t[:3000], y1_remez[:3000], label="Filtered fundamental component", linewidth=2)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (V)")
plt.title("Original signal and equiripple filtered fundamental")
plt.grid(True)
plt.legend()
plt.show()

print(f"Estimated fundamental RMS from equiripple filter = {I1_remez_demo:.6f}")

### 9.5 Comment

The equiripple design offers the most systematic control over the worst case approximation error and often provides the sharpest compromise between passband ripple and stopband attenuation for a given filter length. In this comparison it serves as a useful benchmark against the more transparent first principles designs.

## 10. Comparison of the FIR Design Methods

Up to this point, each FIR design method has been demonstrated by using a representative band pass filter around the fundamental component. The next step is to apply the same harmonic filter bank framework to all three methods so that each method produces its own harmonic rms estimates and corresponding THD value before FFT verification is performed.

### 10.1 Common Harmonic Estimation Framework

For all three design approaches, the same procedure is followed:

1. design one narrow band pass FIR filter for each harmonic $f_n = n f_1$,
2. apply each filter to the measured current by discrete time convolution,
3. calculate the rms value of each filtered output,
4. evaluate

$$
THD = \frac{\sqrt{\sum_{n=2}^{N_h} I_n^2}}{I_1}.
$$

Only the coefficient generation step changes from one method to another.

In [ ]:

def estimate_harmonic_bank(x, f1, fs, max_harmonic, bw, M, method, transition=4.0):
 harmonic_orders = []
 harmonic_rms = []

 for n_harm in range(1, max_harmonic + 1):
 fc = n_harm * f1
 if fc + bw / 2 >= fs / 2:
 break

 if method == "window":
 h = design_bandpass_fir(fc, bw, fs, M)
 elif method == "freq_sampling":
 h = design_bandpass_freq_sampling(fc, bw, fs, M)
 elif method == "remez":
 h = design_bandpass_remez(fc, bw, fs, M, transition=transition)
 else:
 raise ValueError("Unknown method")

 y = np.convolve(x, h, mode='same')
 rms_val = np.sqrt(np.mean(y**2))
 harmonic_orders.append(n_harm)
 harmonic_rms.append(rms_val)

 harmonic_orders = np.array(harmonic_orders)
 harmonic_rms = np.array(harmonic_rms)

 I1 = harmonic_rms[0]
 if I1 == 0:
 thd = np.nan
 else:
 thd = np.sqrt(np.sum(harmonic_rms[1:]**2)) / I1
 return harmonic_orders, harmonic_rms, I1, thd

# Use common settings so that the three FIR methods can be compared fairly
max_harmonic = 15
bw_compare = 6
M_compare = 1201

harmonic_orders, harmonic_rms_window, I1_window, THD_window = estimate_harmonic_bank(
 x_ac, f1, fs, max_harmonic, bw_compare, M_compare, method="window"
)

_, harmonic_rms_freq, I1_freq, THD_freq = estimate_harmonic_bank(
 x_ac, f1, fs, max_harmonic, bw_compare, M_compare, method="freq_sampling"
)

_, harmonic_rms_remez, I1_remez, THD_remez = estimate_harmonic_bank(
 x_ac, f1, fs, max_harmonic, bw_compare, M_compare, method="remez", transition=6.0
)

print(f"Window-method THD estimate = {THD_window:.6f} ({THD_window*100:.4f} %)")
print(f"Frequency-sampling THD estimate = {THD_freq:.6f} ({THD_freq*100:.4f} %)")
print(f"Equiripple / remez THD estimate = {THD_remez:.6f} ({THD_remez*100:.4f} %)")

In [ ]:

import pandas as pd

fir_comparison_df = pd.DataFrame({
 "Method": ["Window", "Frequency sampling", "Equiripple (remez)"],
 "Filter_length": [M_compare, M_compare, M_compare],
 "Bandwidth_Hz": [bw_compare, bw_compare, bw_compare],
 "Fundamental_RMS": [I1_window, I1_freq, I1_remez],
 "THD_percent": [THD_window*100, THD_freq*100, THD_remez*100]
})

fir_comparison_df

In [ ]:

plt.figure(figsize=(12, 5))
plt.bar(harmonic_orders - 0.25, harmonic_rms_window, width=0.25, label="Window")
plt.bar(harmonic_orders, harmonic_rms_freq, width=0.25, label="Freq. sampling")
plt.bar(harmonic_orders + 0.25, harmonic_rms_remez, width=0.25, label="Equiripple")
plt.xlabel("Harmonic number")
plt.ylabel("Estimated RMS")
plt.title("Harmonic RMS estimates from the three FIR design methods")
plt.grid(True, axis='y')
plt.legend()
plt.show()

### 10.2 Comment on the FIR Comparison

This section compares all three FIR techniques on the same basis by keeping the harmonic estimation procedure fixed and changing only the way the filter coefficients are designed. This makes the THD comparison meaningful, since differences in the final result can be linked mainly to the passband shape, stopband attenuation, and transition band behaviour of each FIR design.

This also shows that the three required FIR methods were not just presented separately. Each one was used in the same filter bank framework to estimate the harmonic content of the measured current.

## 11. FFT Verification of the THD Estimates

All three FIR methods can now be verified using the Fast Fourier Transform. The FFT provides an independent frequency domain reference against which the THD estimates from the FIR filter bank can be checked.

### 11.1 FFT Based Harmonic Extraction

The harmonic rms values are extracted from the FFT at integer multiples of the estimated fundamental frequency. The same THD definition is used,

$$
THD = \frac{\sqrt{\sum_{n=2}^{N_h} I_n^2}}{I_1}.
$$

In [ ]:

# Extract harmonic rms values directly from the FFT of the measured signal
N = len(x_ac)
X = np.fft.rfft(x_ac)
f = np.fft.rfftfreq(N, d=dt)

mag = 2 * np.abs(X) / N
mag[0] = np.abs(X[0]) / N

harmonic_rms_fft = []
for n_harm in harmonic_orders:
 target_freq = n_harm * f1
 idx = np.argmin(np.abs(f - target_freq))
 harmonic_rms_fft.append(mag[idx] / np.sqrt(2))

harmonic_rms_fft = np.array(harmonic_rms_fft)
I1_fft = harmonic_rms_fft[0]
THD_fft = np.sqrt(np.sum(harmonic_rms_fft[1:]**2)) / I1_fft

print("FFT harmonic RMS estimates:")
for n_harm, rms_val in zip(harmonic_orders, harmonic_rms_fft):
 print(f"Harmonic {n_harm:2d}: RMS = {rms_val:.6f}")

print(f"\nFundamental RMS from FFT = {I1_fft:.6f}")
print(f"THD from FFT = {THD_fft:.6f}")
print(f"THD from FFT = {THD_fft*100:.4f} %")

In [ ]:

plt.figure(figsize=(12, 5))
plt.bar(harmonic_orders - 0.3, harmonic_rms_window, width=0.2, label="Window")
plt.bar(harmonic_orders - 0.1, harmonic_rms_freq, width=0.2, label="Freq. sampling")
plt.bar(harmonic_orders + 0.1, harmonic_rms_remez, width=0.2, label="Equiripple")
plt.bar(harmonic_orders + 0.3, harmonic_rms_fft, width=0.2, label="FFT")
plt.xlabel("Harmonic number")
plt.ylabel("Estimated RMS")
plt.title("Comparison of harmonic RMS estimates: FIR methods vs FFT")
plt.grid(True, axis='y')
plt.legend()
plt.show()

### 11.2 Calibration of the FFT Implementation

Before accepting the FFT result as the verification benchmark, the FFT code is tested on a synthetic signal whose harmonic content is known exactly. This makes it possible to quantify the numerical error of the FFT based THD calculation directly.

In [ ]:
# Create a synthetic signal with known harmonic content to test the FFT routine
A1 = 1.0
A3 = 0.1
A5 = 0.05
f1_cal = 50.0

t_cal = np.arange(0, 2, dt) # 2 seconds using same sampling interval

x_cal = (
 A1 * np.sin(2 * np.pi * f1_cal * t_cal) +
 A3 * np.sin(2 * np.pi * 3 * f1_cal * t_cal) +
 A5 * np.sin(2 * np.pi * 5 * f1_cal * t_cal)
)

# Calculate the FFT of the synthetic test signal
N_cal = len(x_cal)
X_cal = np.fft.rfft(x_cal)
f_cal = np.fft.rfftfreq(N_cal, d=dt)

mag_cal = 2 * np.abs(X_cal) / N_cal
mag_cal[0] = np.abs(X_cal[0]) / N_cal

harmonic_rms_cal = []
for n_harm in [1, 3, 5]:
 target_freq = n_harm * f1_cal
 idx = np.argmin(np.abs(f_cal - target_freq))
 rms_val = mag_cal[idx] / np.sqrt(2)
 harmonic_rms_cal.append(rms_val)

I1_cal, I3_cal, I5_cal = harmonic_rms_cal
THD_fft_cal = np.sqrt(I3_cal**2 + I5_cal**2) / I1_cal

THD_theoretical = np.sqrt(A3**2 + A5**2) / A1

print(f"Theoretical THD = {THD_theoretical:.6f} ({THD_theoretical*100:.4f}%)")
print(f"FFT-based THD = {THD_fft_cal:.6f} ({THD_fft_cal*100:.4f}%)")
print(f"Absolute error = {abs(THD_fft_cal - THD_theoretical):.6f}")

### 11.3 Comment on the FFT Verification

The calibration result shows that the FFT implementation reproduces the THD of a synthetic signal with known harmonic content with only a very small error. This gives confidence that the FFT method can be used as the verification reference for the measured blocked rotor signal.

The FFT result is not used to replace the FIR filter bank design. Its purpose is to provide an independent benchmark against which the filter bank estimates can be judged. This is the role required by the assignment, where the FIR bank performs the main THD estimation and the FFT provides verification.

## 12. Final Comparison and Discussion

In [ ]:

final_comparison_df = pd.DataFrame({
 "Method": ["Window", "Frequency sampling", "Equiripple (remez)", "FFT"],
 "THD_percent": [THD_window*100, THD_freq*100, THD_remez*100, THD_fft*100]
})

final_comparison_df["Difference_from_FFT_percent"] = final_comparison_df["THD_percent"] - THD_fft*100
final_comparison_df

The final comparison shows that the THD estimate depends on the FIR design method when a practical finite length filter bank is used. Even when the nominal centre frequencies and bandwidths are kept the same, the three designs produce different passband ripple, transition widths, and stopband attenuation. These differences affect the amount of neighbouring spectral energy that leaks into each harmonic estimate and therefore influence the final THD value.

In practice, the FIR based THD estimates are often expected to be somewhat higher than the FFT reference because a real band pass filter cannot isolate each harmonic perfectly. If adjacent spectral content lies inside the passband or inside the transition region, the estimated harmonic rms value may increase. This raises the numerator of the THD expression and leads to a larger result.

The FFT based estimate is less affected by this type of passband leakage when the harmonic frequencies align closely with the spectral bins, which is why it serves as a useful reference in this report. The comparison therefore highlights not only the final THD values, but also the effect that practical FIR design choices have on harmonic separation.

## 13. Conclusion

This report estimated the total harmonic distortion of the measured blocked rotor current by means of a digital FIR filter bank and then verified the result using FFT analysis. Three FIR design approaches from the learning unit were applied within one common harmonic estimation framework: the window method, the frequency sampling method, and the equiripple Parks McClellan method. Each approach was used to construct harmonic filters, extract rms estimates, and produce a corresponding THD value before the FFT verification stage.

A separate calibration experiment showed that the FFT implementation reproduced the theoretical THD of a synthetic signal with very small error. This supports the use of the FFT as a reliable verification method for the measured data.

Overall, the results show that the estimated THD depends on the FIR design method when practical finite length filters are used. The study therefore not only provides a THD estimate for the blocked rotor current, but also demonstrates how filter design choices influence harmonic estimation accuracy.

## References

1. D. Shmilovitz, *On the Definition of Total Harmonic Distortion and Its Effect on Measurement Interpretation*, IEEE Transactions on Power Delivery, vol. 20, no. 1, Jan. 2005. 
2. *Total Harmonic Distortion and Effects in Electrical Power Systems*, Associated Power Technologies.